# Training

### Setup

In [10]:
%reload_ext autoreload
%autoreload 2

In [11]:
# Setup working directory
from pathlib import Path
import os

def find_root_dir(marker='tfg'):
    p = Path.cwd()
    for candidate in [p] + list(p.parents):
        if candidate.name == marker:
            return candidate.resolve()
        if (candidate / marker).is_dir():
            return (candidate / marker).resolve()
    raise FileNotFoundError(f"Could not find '{marker}' folder in {Path.cwd()} or its parents.")

os.chdir(find_root_dir('tfg'))

## Load Datasets and Models

In [12]:
DATASETS = {
    "numeric": "data/processed/stage1/reference-8k-numeric.parquet",
    "encoded": "data/processed/stage1/reference-8k-encoded.parquet",
    "correlation_08": "data/processed/stage1/correlation_08.parquet",
    "correlation_09": "data/processed/stage1/correlation_09.parquet",
}

In [13]:
from src.models import get_baseline_models
models = get_baseline_models()

In [14]:
from src.preprocessing import load_dataset, split_features_target, encode_target

X_SETS = {}
Y_SETS = {}

for feature_set_name, path in DATASETS.items():

    df = load_dataset(path)
    X, y = split_features_target(df,target_column="Label")
    y = encode_target(y)

    X_SETS[feature_set_name] = X
    Y_SETS[feature_set_name] = y

In [15]:
for name, X in X_SETS.items():
    print(f"{name}: {X.shape}")

numeric: (7997, 67)
encoded: (7997, 78)
correlation_08: (7997, 35)
correlation_09: (7997, 41)


## Train

In [16]:
from src.train import train
saved = train(
    X_SETS,
    Y_SETS,
    models,
    dir_name="models/stage1",
    scaler=None,
    test_size=0.2
)

Training models on every feature set

========== Logistic Regression (models\stage1\Logistic Regression) ==========

Preparing Logistic Regression with numeric...
Train accuracy: 0.9456 | Train F1: 0.9455

Preparing Logistic Regression with encoded...
Train accuracy: 0.9445 | Train F1: 0.9444

Preparing Logistic Regression with correlation_08...
Train accuracy: 0.9870 | Train F1: 0.9870

Preparing Logistic Regression with correlation_09...
Train accuracy: 0.9647 | Train F1: 0.9647

========== SVC (models\stage1\SVC) ==========

Preparing SVC with numeric...
Train accuracy: 0.8849 | Train F1: 0.8849

Preparing SVC with encoded...
Train accuracy: 0.8849 | Train F1: 0.8849

Preparing SVC with correlation_08...
Train accuracy: 0.8796 | Train F1: 0.8794

Preparing SVC with correlation_09...
Train accuracy: 0.8801 | Train F1: 0.8799

========== MLP (models\stage1\MLP) ==========

Preparing MLP with numeric...
Train accuracy: 0.9873 | Train F1: 0.9873

Preparing MLP with encoded...
Train accu

## Evaluation

In [17]:
from src.evaluation import evaluate_models, save_results
results = evaluate_models(saved)
save_results(results, "results/", "results_stage_I.json")

Evaluating Logistic Regression on numeric...
Evaluating Logistic Regression on encoded...
Evaluating Logistic Regression on correlation_08...
Evaluating Logistic Regression on correlation_09...
Evaluating SVC on numeric...
Evaluating SVC on encoded...
Evaluating SVC on correlation_08...
Evaluating SVC on correlation_09...
Evaluating MLP on numeric...
Evaluating MLP on encoded...
Evaluating MLP on correlation_08...
Evaluating MLP on correlation_09...
Results saved to: C:\Users\Monon\university\4th-year-bachelor\tfg\results\results_stage_I.json


In [18]:
results

,Model,Feature Set,Features Count,Train Time,Train Accuracy,Train F1,BENIGN,DDoS,FP,FN,TP,TN,FP Rate,Accuracy,F1,Features Names
0,Logistic Regression,numeric,67,2.056,0.945599,0.945501,800,800,93,6,794,707,0.11625,0.938125,0.937942,"[Flow Duration, Total Fwd Packets, Total Backw..."
1,Logistic Regression,encoded,78,2.069,0.944505,0.944419,800,800,91,11,789,709,0.11375,0.936250,0.936090,"[Source IP, Destination IP, Flow Duration, Tot..."
2,Logistic Regression,correlation_08,35,2.155,0.987025,0.987025,800,800,7,14,786,793,0.00875,0.986875,0.986875,"[Source IP, Destination IP, Fwd Packet Length ..."
3,Logistic Regression,correlation_09,41,2.056,0.964671,0.964665,800,800,33,16,784,767,0.04125,0.969375,0.969372,"[Source IP, Destination IP, Fwd Packet Length ..."
4,SVC,numeric,67,0.570,0.884946,0.884920,800,800,84,87,713,716,0.10500,0.893125,0.893125,"[Flow Duration, Total Fwd Packets, Total Backw..."
5,SVC,encoded,78,0.599,0.884946,0.884920,800,800,84,87,713,716,0.10500,0.893125,0.893125,"[Source IP, Destination IP, Flow Duration, Tot..."
6,SVC,correlation_08,35,0.507,0.879631,0.879439,800,800,73,126,674,727,0.09125,0.875625,0.875488,"[Source IP, Destination IP, Fwd Packet Length ..."
7,SVC,correlation_09,41,0.501,0.880100,0.879904,800,800,74,126,674,726,0.09250,0.875000,0.874868,"[Source IP, Destination IP, Fwd Packet Length ..."
8,MLP,numeric,67,0.850,0.987338,0.987337,800,800,18,5,795,782,0.02250,0.985625,0.985624,"[Flow Duration, Total Fwd Packets, Total Backw..."
9,MLP,encoded,78,0.651,0.980460,0.980455,800,800,26,6,794,774,0.03250,0.980000,0.979997,"[Source IP, Destination IP, Flow Duration, Tot..."
